# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 9

In this version, we extend the scaled dot-product attention from Version 8 into a first Transformer block.

The model keeps the same character vocabulary, fixed four-character context, trainable character embeddings and positional embeddings used in the previous version.

Version 8 allowed every position to attend to every other position in the context.

Version 9 introduces causal self-attention so that each position can attend only to itself and to earlier positions.

The training targets also become sequential: every input position learns to predict the character that follows it.

For example:

`mode`  
↓  
`odel`

This allows the model to learn next-character prediction at every position while respecting the causal direction required for autoregressive language modeling.

The attention output is then combined with its input through a residual connection and normalized.

A small feed-forward network processes each position independently, followed by a second residual connection and normalization step.

The resulting sequence of operations forms a single Transformer block:

`character + positional embeddings`  
↓  
`causal self-attention`  
↓  
`residual connection + normalization`  
↓  
`feed-forward network`  
↓  
`residual connection + normalization`

The contextual representation at every position produces next-character logits during training.

During generation, the final position is still used to predict and sample the next character.

This version introduces the structure of a Transformer block while keeping the model small, explicit and easy to inspect.

## 1. Imports and Configuration

The Python standard library configures the execution environment before TensorFlow is imported.

GPU execution is disabled because this small model runs efficiently on the CPU and does not require CUDA. Low-level TensorFlow logs are suppressed to keep the notebook output clean.

TensorFlow provides tensor operations, trainable variables and automatic differentiation.

NumPy remains useful for reproducible data shuffling and sampling, while TensorFlow performs the model calculations and training.

The configuration collects the values that control the experiment.

`CONTEXT_LENGTH` defines the number of character positions processed together.

`EMBEDDING_DIM` defines the size of the trainable character and positional representations.

`ATTENTION_DIM` defines the size of the query, key and value vectors.

`FEED_FORWARD_DIM` defines the internal size of the feed-forward network inside the Transformer block.

`LAYER_NORM_EPSILON` provides numerical stability when normalizing the representations.

An explicit seed makes weight initialization, data splitting, mini-batch shuffling and text generation reproducible.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import tensorflow as tf

tf.config.set_visible_devices([], "GPU")

In [2]:
SEED = 42
TRAIN_FRACTION = 0.8
BATCH_SIZE = 32
LEARNING_RATE = 1.0
EPOCHS = 100

CONTEXT_LENGTH = 4
EMBEDDING_DIM = 8
ATTENTION_DIM = 16
FEED_FORWARD_DIM = 16
LAYER_NORM_EPSILON = 1e-5

tf.keras.utils.set_random_seed(SEED)

## 2. Training Data

### Training text

The English corpus is included directly in the notebook. It provides the text from which the model learns character patterns.

The corpus is kept unchanged from Version 8 so that the effect of introducing a Transformer block can be observed without changing the training data.

In [3]:
corpus = 'language models learn patterns from text.\na small model predicts what character may come next.\nwe begin with counting because counting is easy to inspect.\nthe model sees letters, spaces, and punctuation.\neach prediction comes from examples found in the training text.\nsimple systems help us understand more advanced systems.\nlater versions will learn parameters with neural networks.\nclear experiments make machine learning easier to study.'

print(corpus)
print("Characters:", len(corpus))

language models learn patterns from text.
a small model predicts what character may come next.
we begin with counting because counting is easy to inspect.
the model sees letters, spaces, and punctuation.
each prediction comes from examples found in the training text.
simple systems help us understand more advanced systems.
later versions will learn parameters with neural networks.
clear experiments make machine learning easier to study.
Characters: 440


### Vocabulary

The vocabulary is the set of symbols the model can represent. Because this is a character model, every letter, space, punctuation mark and newline is a token.

Each character is assigned an integer identifier.

As in Version 8, these identifiers are used to look up trainable embedding vectors.

In [4]:
vocabulary = sorted(set(corpus))
vocabulary_size = len(vocabulary)

character_to_id = {character: index for index, character in enumerate(vocabulary)}
id_to_character = {index: character for character, index in character_to_id.items()}

print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", repr("".join(vocabulary)))
print("First mappings:", list(character_to_id.items())[:10])

Vocabulary size: 27
Vocabulary: '\n ,.abcdefghiklmnoprstuvwxy'
First mappings: [('\n', 0), (' ', 1), (',', 2), ('.', 3), ('a', 4), ('b', 5), ('c', 6), ('d', 7), ('e', 8), ('f', 9)]


### Context windows and numerical encoding

Each character is converted into its integer identifier.

For the text `modeling`:

`modeling`  
↓  
`[15, 17, 7, 8, 14, 12, 16, 10]`

The model keeps the fixed four-character context used in Version 8.

Version 9 changes the training targets.

Instead of predicting only one character after the complete context, every input position now predicts the character that immediately follows it.

For example:

`mode`  
↓  
`odel`

This creates four next-character predictions:

- `m -> o`
- `mo -> d`
- `mod -> e`
- `mode -> l`

Numerically:

`[15, 17, 7, 8]`  
↓  
`[17, 7, 8, 14]`

The input and target therefore both contain four character identifiers.

A causal attention mask ensures that each position can use only itself and earlier input positions when making its prediction.

The final position still performs the same task used in Version 8: the complete four-character context predicts the following character.

During generation, only the prediction produced by the final position is sampled before the context window moves forward.

In [5]:
examples = [
    (
        corpus[index:index + CONTEXT_LENGTH],
        corpus[index + 1:index + CONTEXT_LENGTH + 1]
    )
    for index in range(len(corpus) - CONTEXT_LENGTH)
]

input_ids = np.array([
    [character_to_id[character] for character in context]
    for context, _ in examples
], dtype=np.int32)

target_ids = np.array([
    [character_to_id[character] for character in targets]
    for _, targets in examples
], dtype=np.int32)

print("Number of examples:", len(examples))
print("Input shape:", input_ids.shape)
print("Target shape:", target_ids.shape)
print("First 8 examples:", examples[:8])
print("First 8 input IDs:")
print(input_ids[:8])
print("First 8 target IDs:")
print(target_ids[:8])

Number of examples: 436
Input shape: (436, 4)
Target shape: (436, 4)
First 8 examples: [('lang', 'angu'), ('angu', 'ngua'), ('ngua', 'guag'), ('guag', 'uage'), ('uage', 'age '), ('age ', 'ge m'), ('ge m', 'e mo'), ('e mo', ' mod')]
First 8 input IDs:
[[14  4 16 10]
 [ 4 16 10 22]
 [16 10 22  4]
 [10 22  4 10]
 [22  4 10  8]
 [ 4 10  8  1]
 [10  8  1 15]
 [ 8  1 15 17]]
First 8 target IDs:
[[ 4 16 10 22]
 [16 10 22  4]
 [10 22  4 10]
 [22  4 10  8]
 [ 4 10  8  1]
 [10  8  1 15]
 [ 8  1 15 17]
 [ 1 15 17  7]]


## 3. Neural Model

### Trainable embeddings and Transformer parameters

Version 9 extends the scaled dot-product attention from Version 8 into a single Transformer block.

Each character identifier still selects a trainable character embedding, and a trainable positional embedding is added to preserve the order of the four context positions.

The positioned representations are projected into queries, keys and values.

A causal mask is applied to the attention scores before softmax. This prevents each position from using information from later positions.

The attention output initially has `ATTENTION_DIM` values per position. A trainable output projection maps it back to `EMBEDDING_DIM` values so that it can be added to the original positioned representation through a residual connection.

The result is normalized with layer normalization.

A feed-forward network then processes every position independently:

`EMBEDDING_DIM`  
↓  
`FEED_FORWARD_DIM`  
↓  
`EMBEDDING_DIM`

A second residual connection and layer normalization complete the Transformer block.

Finally, every position is projected to vocabulary logits so that each position learns to predict the following character.

The block therefore follows this sequence:

`character + positional embeddings`  
↓  
`Q / K / V projections`  
↓  
`causal scaled dot-product attention`  
↓  
`attention output projection`  
↓  
`residual connection + layer normalization`  
↓  
`feed-forward network`  
↓  
`residual connection + layer normalization`  
↓  
`next-character logits for every position`

The implementation still uses one attention mechanism rather than multi-head attention, keeping the first Transformer block explicit and easy to inspect.

In [6]:
inputs = tf.convert_to_tensor(input_ids, dtype=tf.int32)
targets = tf.convert_to_tensor(target_ids, dtype=tf.int32)

model_random = tf.random.Generator.from_seed(SEED)

embedding_matrix = tf.Variable(
    model_random.normal(
        shape=(vocabulary_size, EMBEDDING_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

position_embedding_matrix = tf.Variable(
    model_random.normal(
        shape=(CONTEXT_LENGTH, EMBEDDING_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

query_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, ATTENTION_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

key_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, ATTENTION_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

value_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, ATTENTION_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

attention_output_weights = tf.Variable(
    model_random.normal(
        shape=(ATTENTION_DIM, EMBEDDING_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

layer_norm_1_scale = tf.Variable(tf.ones((EMBEDDING_DIM,), dtype=tf.float32))
layer_norm_1_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,), dtype=tf.float32))

feed_forward_input_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, FEED_FORWARD_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

feed_forward_output_weights = tf.Variable(
    model_random.normal(
        shape=(FEED_FORWARD_DIM, EMBEDDING_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

layer_norm_2_scale = tf.Variable(tf.ones((EMBEDDING_DIM,), dtype=tf.float32))
layer_norm_2_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,), dtype=tf.float32))

output_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, vocabulary_size),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)


def layer_normalize(values, scale, shift):
    mean = tf.reduce_mean(values, axis=-1, keepdims=True)
    variance = tf.reduce_mean(tf.square(values - mean), axis=-1, keepdims=True)
    normalized = (values - mean) / tf.sqrt(variance + LAYER_NORM_EPSILON)

    return normalized * scale + shift


def transformer_forward(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    attention_output_weights,
    layer_norm_1_scale,
    layer_norm_1_shift,
    feed_forward_input_weights,
    feed_forward_output_weights,
    layer_norm_2_scale,
    layer_norm_2_shift,
    output_weights,
    inputs
):
    embeddings = tf.gather(embedding_matrix, inputs)
    positioned_embeddings = embeddings + position_embedding_matrix[tf.newaxis, :, :]

    queries = tf.matmul(positioned_embeddings, query_weights)
    keys = tf.matmul(positioned_embeddings, key_weights)
    values = tf.matmul(positioned_embeddings, value_weights)

    attention_scores = tf.matmul(queries, keys, transpose_b=True)

    scale = tf.sqrt(tf.cast(ATTENTION_DIM, tf.float32))
    scaled_attention_scores = attention_scores / scale

    causal_mask = tf.linalg.band_part(
        tf.ones((CONTEXT_LENGTH, CONTEXT_LENGTH), dtype=tf.float32),
        -1,
        0
    )

    masked_attention_scores = tf.where(
        causal_mask == 1.0,
        scaled_attention_scores,
        tf.constant(-1e9, dtype=tf.float32)
    )

    attention_weights = tf.nn.softmax(masked_attention_scores, axis=-1)
    attention_output = tf.matmul(attention_weights, values)

    projected_attention = tf.matmul(attention_output, attention_output_weights)

    attention_residual = positioned_embeddings + projected_attention
    normalized_attention = layer_normalize(
        attention_residual,
        layer_norm_1_scale,
        layer_norm_1_shift
    )

    feed_forward_hidden = tf.nn.relu(
        tf.matmul(normalized_attention, feed_forward_input_weights)
    )
    feed_forward_output = tf.matmul(
        feed_forward_hidden,
        feed_forward_output_weights
    )

    feed_forward_residual = normalized_attention + feed_forward_output
    transformer_output = layer_normalize(
        feed_forward_residual,
        layer_norm_2_scale,
        layer_norm_2_shift
    )

    logits = tf.matmul(transformer_output, output_weights)

    return transformer_output, attention_weights, logits

In [7]:
example_embeddings = tf.gather(embedding_matrix, inputs[:1])

example_positioned_embeddings = (example_embeddings + position_embedding_matrix[tf.newaxis, :, :])

example_queries = tf.matmul(example_positioned_embeddings, query_weights)
example_keys = tf.matmul(example_positioned_embeddings, key_weights)
example_values = tf.matmul(example_positioned_embeddings, value_weights)

causal_mask = tf.linalg.band_part(tf.ones((CONTEXT_LENGTH, CONTEXT_LENGTH), dtype=tf.float32), -1, 0)

example_transformer_output, example_attention_weights, example_logits = transformer_forward(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    attention_output_weights,
    layer_norm_1_scale,
    layer_norm_1_shift,
    feed_forward_input_weights,
    feed_forward_output_weights,
    layer_norm_2_scale,
    layer_norm_2_shift,
    output_weights,
    inputs[:1]
)

trainable_parameters = (
    tf.size(embedding_matrix)
    + tf.size(position_embedding_matrix)
    + tf.size(query_weights)
    + tf.size(key_weights)
    + tf.size(value_weights)
    + tf.size(attention_output_weights)
    + tf.size(layer_norm_1_scale)
    + tf.size(layer_norm_1_shift)
    + tf.size(feed_forward_input_weights)
    + tf.size(feed_forward_output_weights)
    + tf.size(layer_norm_2_scale)
    + tf.size(layer_norm_2_shift)
    + tf.size(output_weights)
)

print("Input tensor shape:", inputs.shape)
print("Target tensor shape:", targets.shape)
print("Embedding matrix shape:", embedding_matrix.shape)
print("Position embedding matrix shape:", position_embedding_matrix.shape)
print("Positioned context shape:", example_positioned_embeddings.shape)

print("Query weight matrix shape:", query_weights.shape)
print("Key weight matrix shape:", key_weights.shape)
print("Value weight matrix shape:", value_weights.shape)

print("Queries shape:", example_queries.shape)
print("Keys shape:", example_keys.shape)
print("Values shape:", example_values.shape)

print("Causal mask:")
print(causal_mask.numpy())

print("Attention weights shape:", example_attention_weights.shape)
print("Transformer output shape:", example_transformer_output.shape)

print("Attention output projection shape:", attention_output_weights.shape)
print("Feed-forward input weights shape:", feed_forward_input_weights.shape)
print("Feed-forward output weights shape:", feed_forward_output_weights.shape)
print("Output weight matrix shape:", output_weights.shape)

print("Logits shape:", example_logits.shape)
print("Trainable parameters:", trainable_parameters.numpy())

Input tensor shape: (436, 4)
Target tensor shape: (436, 4)
Embedding matrix shape: (27, 8)
Position embedding matrix shape: (4, 8)
Positioned context shape: (1, 4, 8)
Query weight matrix shape: (8, 16)
Key weight matrix shape: (8, 16)
Value weight matrix shape: (8, 16)
Queries shape: (1, 4, 16)
Keys shape: (1, 4, 16)
Values shape: (1, 4, 16)
Causal mask:
[[1. 0. 0. 0.]
 [1. 1. 0. 0.]
 [1. 1. 1. 0.]
 [1. 1. 1. 1.]]
Attention weights shape: (1, 4, 4)
Transformer output shape: (1, 4, 8)
Attention output projection shape: (16, 8)
Feed-forward input weights shape: (8, 16)
Feed-forward output weights shape: (16, 8)
Output weight matrix shape: (8, 27)
Logits shape: (1, 4, 27)
Trainable parameters: 1264


### Training and validation split

The examples are divided into two separate groups:

- the training set is used to update the model parameters;
- the validation set is used to measure the loss on examples that do not update the parameters.

Each input example contains four character identifiers and each target contains the four following character identifiers.

The same reproducible train/validation split used in Version 8 is preserved.

Inside the model, causal attention ensures that every prediction can use only the current and previous input positions.

The indices are shuffled with a local random generator, making the split reproducible.

In [8]:
split_random = np.random.default_rng(SEED)

indices = split_random.permutation(len(inputs))
split_position = int(len(indices) * TRAIN_FRACTION)

train_indices = indices[:split_position]
validation_indices = indices[split_position:]

train_inputs = tf.gather(inputs, train_indices)
train_targets = tf.gather(targets, train_indices)

validation_inputs = tf.gather(inputs, validation_indices)
validation_targets = tf.gather(targets, validation_indices)

print("Training examples:", len(train_inputs))
print("Validation examples:", len(validation_inputs))
print("Training input shape:", train_inputs.shape)
print("Training target shape:", train_targets.shape)
print("Validation input shape:", validation_inputs.shape)
print("Validation target shape:", validation_targets.shape)

Training examples: 348
Validation examples: 88
Training input shape: (348, 4)
Training target shape: (348, 4)
Validation input shape: (88, 4)
Validation target shape: (88, 4)


### Softmax probabilities

TensorFlow provides `tf.nn.softmax` to convert logits into probabilities.

The Transformer produces one vocabulary distribution for every context position.

For each position:

- every probability is between 0 and 1;
- the probabilities over the vocabulary sum to 1;
- higher logits produce higher probabilities.

Softmax is therefore applied along the final vocabulary dimension.

TensorFlow handles the numerical stability of this operation internally.

In [9]:
def softmax(logits):
    return tf.nn.softmax(logits, axis=-1)

### Cross-entropy loss

Cross-entropy measures how much probability the model assigns to the correct next character.

The Transformer produces next-character logits for all four positions.

For an input such as:

`mode`

the corresponding targets are:

`odel`

The loss therefore measures four predictions instead of only the prediction from the final position.

TensorFlow calculates sparse softmax cross-entropy directly from the logits and integer target IDs using a numerically stable operation.

The individual losses have one value for every example and every context position.

Their mean produces one scalar loss used for training.

Gradients flow through the output projection, both normalization steps, the feed-forward network, residual connections, causal attention, Q/K/V projections, positional embeddings and character embeddings.

In [10]:
def calculate_loss(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    attention_output_weights,
    layer_norm_1_scale,
    layer_norm_1_shift,
    feed_forward_input_weights,
    feed_forward_output_weights,
    layer_norm_2_scale,
    layer_norm_2_shift,
    output_weights,
    inputs,
    targets
):
    _, _, logits = transformer_forward(
        embedding_matrix,
        position_embedding_matrix,
        query_weights,
        key_weights,
        value_weights,
        attention_output_weights,
        layer_norm_1_scale,
        layer_norm_1_shift,
        feed_forward_input_weights,
        feed_forward_output_weights,
        layer_norm_2_scale,
        layer_norm_2_shift,
        output_weights,
        inputs
    )

    example_losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)

    return tf.reduce_mean(example_losses)

In [11]:
initial_train_loss = calculate_loss(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    attention_output_weights,
    layer_norm_1_scale,
    layer_norm_1_shift,
    feed_forward_input_weights,
    feed_forward_output_weights,
    layer_norm_2_scale,
    layer_norm_2_shift,
    output_weights,
    train_inputs,
    train_targets
)

initial_validation_loss = calculate_loss(
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    attention_output_weights,
    layer_norm_1_scale,
    layer_norm_1_shift,
    feed_forward_input_weights,
    feed_forward_output_weights,
    layer_norm_2_scale,
    layer_norm_2_shift,
    output_weights,
    validation_inputs,
    validation_targets
)

print("Initial train loss:", initial_train_loss.numpy())
print("Initial validation loss:", initial_validation_loss.numpy())

Initial train loss: 3.29802
Initial validation loss: 3.299714


### Training with mini-batch gradient descent

Training remains organized into epochs.

At each recorded epoch, training and validation loss are measured using the current model parameters.

Except at the final recorded epoch, the training examples are then shuffled and divided into mini-batches.

For every mini-batch:

1. `tf.GradientTape` records the complete Transformer forward computation;
2. TensorFlow calculates the gradients automatically;
3. all trainable parameters are updated manually with gradient descent.

The validation examples are never used to update the model parameters.

Gradients flow backward through the vocabulary output projection, the second normalization step, the feed-forward network, the first normalization step, the attention output projection, causal attention, query-key-value projections, positional embeddings and character embeddings.

The same Transformer parameters are reused at every position in the context.

Version 9 keeps the explicit mini-batch training procedure while extending the attention model into a complete single Transformer block.

### Mini-batches

A mini-batch is a small group of training examples.

The model updates its parameters after every mini-batch instead of processing all training examples together.

The final mini-batch may contain fewer examples than the configured batch size.

In [12]:
def create_batches(inputs, targets, batch_size):
    for start in range(0, len(inputs), batch_size):
        end = start + batch_size

        batch_inputs = inputs[start:end]
        batch_targets = targets[start:end]

        yield batch_inputs, batch_targets

In [13]:
def train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    initial_embedding_matrix,
    initial_position_embedding_matrix,
    initial_query_weights,
    initial_key_weights,
    initial_value_weights,
    initial_attention_output_weights,
    initial_layer_norm_1_scale,
    initial_layer_norm_1_shift,
    initial_feed_forward_input_weights,
    initial_feed_forward_output_weights,
    initial_layer_norm_2_scale,
    initial_layer_norm_2_shift,
    initial_output_weights,
    learning_rate=1.0,
    batch_size=32,
    epochs=100,
    seed=42,
    print_every=10
):
    trained_embedding_matrix = tf.Variable(initial_embedding_matrix)
    trained_position_embedding_matrix = tf.Variable(initial_position_embedding_matrix)
    trained_query_weights = tf.Variable(initial_query_weights)
    trained_key_weights = tf.Variable(initial_key_weights)
    trained_value_weights = tf.Variable(initial_value_weights)
    trained_attention_output_weights = tf.Variable(initial_attention_output_weights)
    trained_layer_norm_1_scale = tf.Variable(initial_layer_norm_1_scale)
    trained_layer_norm_1_shift = tf.Variable(initial_layer_norm_1_shift)
    trained_feed_forward_input_weights = tf.Variable(initial_feed_forward_input_weights)
    trained_feed_forward_output_weights = tf.Variable(initial_feed_forward_output_weights)
    trained_layer_norm_2_scale = tf.Variable(initial_layer_norm_2_scale)
    trained_layer_norm_2_shift = tf.Variable(initial_layer_norm_2_shift)
    trained_output_weights = tf.Variable(initial_output_weights)

    training_random = np.random.default_rng(seed)

    train_loss_history = []
    validation_loss_history = []

    for epoch in range(epochs + 1):
        train_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_position_embedding_matrix,
                trained_query_weights,
                trained_key_weights,
                trained_value_weights,
                trained_attention_output_weights,
                trained_layer_norm_1_scale,
                trained_layer_norm_1_shift,
                trained_feed_forward_input_weights,
                trained_feed_forward_output_weights,
                trained_layer_norm_2_scale,
                trained_layer_norm_2_shift,
                trained_output_weights,
                train_inputs,
                train_targets
            ).numpy()
        )

        validation_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_position_embedding_matrix,
                trained_query_weights,
                trained_key_weights,
                trained_value_weights,
                trained_attention_output_weights,
                trained_layer_norm_1_scale,
                trained_layer_norm_1_shift,
                trained_feed_forward_input_weights,
                trained_feed_forward_output_weights,
                trained_layer_norm_2_scale,
                trained_layer_norm_2_shift,
                trained_output_weights,
                validation_inputs,
                validation_targets
            ).numpy()
        )

        train_loss_history.append(train_loss)
        validation_loss_history.append(validation_loss)

        if print_every is not None and epoch % print_every == 0:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {train_loss:.4f} | "
                f"Validation loss: {validation_loss:.4f}"
            )

        if epoch == epochs:
            break

        shuffled_indices = training_random.permutation(len(train_inputs))

        shuffled_inputs = tf.gather(train_inputs, shuffled_indices)
        shuffled_targets = tf.gather(train_targets, shuffled_indices)

        for batch_inputs, batch_targets in create_batches(shuffled_inputs, shuffled_targets, batch_size):
            with tf.GradientTape() as tape:
                batch_loss = calculate_loss(
                    trained_embedding_matrix,
                    trained_position_embedding_matrix,
                    trained_query_weights,
                    trained_key_weights,
                    trained_value_weights,
                    trained_attention_output_weights,
                    trained_layer_norm_1_scale,
                    trained_layer_norm_1_shift,
                    trained_feed_forward_input_weights,
                    trained_feed_forward_output_weights,
                    trained_layer_norm_2_scale,
                    trained_layer_norm_2_shift,
                    trained_output_weights,
                    batch_inputs,
                    batch_targets
                )

            trainable_variables = [
                trained_embedding_matrix,
                trained_position_embedding_matrix,
                trained_query_weights,
                trained_key_weights,
                trained_value_weights,
                trained_attention_output_weights,
                trained_layer_norm_1_scale,
                trained_layer_norm_1_shift,
                trained_feed_forward_input_weights,
                trained_feed_forward_output_weights,
                trained_layer_norm_2_scale,
                trained_layer_norm_2_shift,
                trained_output_weights
            ]

            gradients = tape.gradient(batch_loss, trainable_variables)

            gradients[0] = tf.convert_to_tensor(gradients[0])

            for variable, gradient in zip(trainable_variables, gradients):
                variable.assign_sub(learning_rate * gradient)

    return (
        trained_embedding_matrix,
        trained_position_embedding_matrix,
        trained_query_weights,
        trained_key_weights,
        trained_value_weights,
        trained_attention_output_weights,
        trained_layer_norm_1_scale,
        trained_layer_norm_1_shift,
        trained_feed_forward_input_weights,
        trained_feed_forward_output_weights,
        trained_layer_norm_2_scale,
        trained_layer_norm_2_shift,
        trained_output_weights,
        train_loss_history,
        validation_loss_history
    )

In [14]:
(
    trained_embedding_matrix,
    trained_position_embedding_matrix,
    trained_query_weights,
    trained_key_weights,
    trained_value_weights,
    trained_attention_output_weights,
    trained_layer_norm_1_scale,
    trained_layer_norm_1_shift,
    trained_feed_forward_input_weights,
    trained_feed_forward_output_weights,
    trained_layer_norm_2_scale,
    trained_layer_norm_2_shift,
    trained_output_weights,
    train_loss_history,
    validation_loss_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    attention_output_weights,
    layer_norm_1_scale,
    layer_norm_1_shift,
    feed_forward_input_weights,
    feed_forward_output_weights,
    layer_norm_2_scale,
    layer_norm_2_shift,
    output_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED
)

final_train_loss = train_loss_history[-1]
final_validation_loss = validation_loss_history[-1]

best_validation_epoch = int(np.argmin(validation_loss_history))
best_validation_loss = validation_loss_history[best_validation_epoch]

print()
print("Initial train loss:", initial_train_loss.numpy())
print("Final train loss:", final_train_loss)
print("Initial validation loss:", initial_validation_loss.numpy())
print("Final validation loss:", final_validation_loss)
print("Best validation loss:", best_validation_loss)
print("Best validation epoch:", best_validation_epoch)

Epoch   0 | Train loss: 3.2980 | Validation loss: 3.2997
Epoch  10 | Train loss: 2.1267 | Validation loss: 2.2967
Epoch  20 | Train loss: 2.0553 | Validation loss: 2.2553
Epoch  30 | Train loss: 2.0050 | Validation loss: 2.2107
Epoch  40 | Train loss: 2.0285 | Validation loss: 2.2150
Epoch  50 | Train loss: 2.0484 | Validation loss: 2.2412
Epoch  60 | Train loss: 1.9716 | Validation loss: 2.1874
Epoch  70 | Train loss: 2.0087 | Validation loss: 2.2071
Epoch  80 | Train loss: 1.9171 | Validation loss: 2.1954
Epoch  90 | Train loss: 1.9634 | Validation loss: 2.2505
Epoch 100 | Train loss: 1.9159 | Validation loss: 2.2090

Initial train loss: 3.29802
Final train loss: 1.9158830642700195
Initial validation loss: 3.299714
Final validation loss: 2.208963394165039
Best validation loss: 2.130791425704956
Best validation epoch: 67


### Reading the losses

At initialization, training and validation loss are close to the value expected from an almost uniform probability distribution over the 27-character vocabulary.

During training, the training loss decreases substantially, showing that the Transformer learns next-character relationships from the corpus.

Validation loss also improves, reaches a minimum during training and then fluctuates while training loss continues to decrease.

This behavior suggests mild overfitting after the best validation point.

The exact loss values and best validation epoch are calculated and printed by the training cell above. Small numerical differences may appear between complete executions depending on the TensorFlow execution environment.

Version 9 also uses a different training objective from Version 8: every context position contributes to the loss instead of only the final position. The loss values of the two versions should therefore not be interpreted as a direct architecture-to-architecture comparison.

The results show that the causal Transformer block successfully learns next-character relationships while respecting the autoregressive direction of the sequence.

### Learned probabilities

After training, the Transformer produces a next-character probability distribution for every position in the context.

Character embeddings and positional embeddings form the initial representations.

Causal self-attention allows each position to combine information only from itself and earlier positions.

The attention result passes through residual connections, layer normalization and the feed-forward network before producing vocabulary logits.

During training, all four positions contribute to the loss.

During generation, only the probability distribution produced by the final position is used to sample the next character.

The example below inspects the final-position next-character distribution for the context `mode`.

In [15]:
example_context = "mode"

example_context_ids = tf.constant(
    [[character_to_id[character] for character in example_context]],
    dtype=tf.int32
)

(
    example_transformer_output,
    example_attention_weights,
    example_logits
) = transformer_forward(
    trained_embedding_matrix,
    trained_position_embedding_matrix,
    trained_query_weights,
    trained_key_weights,
    trained_value_weights,
    trained_attention_output_weights,
    trained_layer_norm_1_scale,
    trained_layer_norm_1_shift,
    trained_feed_forward_input_weights,
    trained_feed_forward_output_weights,
    trained_layer_norm_2_scale,
    trained_layer_norm_2_shift,
    trained_output_weights,
    example_context_ids
)

learned_probabilities = softmax(example_logits)[0, -1].numpy()

sorted_probabilities = sorted(
    zip(vocabulary, learned_probabilities),
    key=lambda item: item[1],
    reverse=True
)

print("Context:", repr(example_context))
print("Transformer output shape:", example_transformer_output.shape)
print("Attention weights shape:", example_attention_weights.shape)
print("Logits shape:", example_logits.shape)
print()

for character, probability in sorted_probabilities:
    print(repr(character), round(float(probability), 4))

print()
print("Total probability:", learned_probabilities.sum())

Context: 'mode'
Transformer output shape: (1, 4, 8)
Attention weights shape: (1, 4, 4)
Logits shape: (1, 4, 27)

' ' 0.3648
'r' 0.1843
'd' 0.0762
'l' 0.0639
't' 0.0583
'm' 0.0465
's' 0.045
'u' 0.0403
'a' 0.0271
'n' 0.0267
'c' 0.0174
'g' 0.017
'x' 0.0123
'e' 0.0077
'i' 0.004
'p' 0.0038
'.' 0.0032
',' 0.0008
'b' 0.0004
'w' 0.0002
'f' 0.0
'k' 0.0
'o' 0.0
'y' 0.0
'h' 0.0
'v' 0.0
'\n' 0.0

Total probability: 1.0000001


### Inspecting causal attention weights

The Transformer produces an explicit `4 x 4` causal attention matrix.

Each row corresponds to one query position and each column corresponds to one key position.

Unlike Version 8, positions are not allowed to attend to future positions.

The causal mask therefore forces all attention weights above the main diagonal to zero.

For a four-character context:

- the first position can attend only to itself;
- the second position can attend to the first two positions;
- the third position can attend to the first three positions;
- the final position can attend to all four positions.

Every row still sums to 1 after softmax.

The example below displays the causal attention weights learned for the context `mode`.

In [16]:
print("Context positions:", list(example_context))
print()

print("Causal attention weights:")
print(np.round(example_attention_weights[0].numpy(), 4))

print()
print("Final-position attention:")

for character, weight in zip(
    example_context,
    example_attention_weights[0, -1].numpy()
):
    print(repr(character), round(float(weight), 4))

print()
print(
    "Final row sum:",
    example_attention_weights[0, -1].numpy().sum()
)

Context positions: ['m', 'o', 'd', 'e']

Causal attention weights:
[[1.     0.     0.     0.    ]
 [0.9234 0.0766 0.     0.    ]
 [0.2832 0.3961 0.3207 0.    ]
 [0.1037 0.4917 0.2196 0.1849]]

Final-position attention:
'm' 0.1037
'o' 0.4917
'd' 0.2196
'e' 0.1849

Final row sum: 0.9999999


## 4. Generator

The trained Transformer can now generate new text one character at a time.

Generation remains autoregressive.

For every prediction:

1. the four most recent characters form the current context;
2. character and positional embeddings create the initial representations;
3. causal self-attention combines information without allowing access to future positions;
4. the attention result passes through the first residual connection and layer normalization;
5. the feed-forward network processes each position;
6. the second residual connection and layer normalization complete the Transformer block;
7. the logits from the final position are converted into probabilities;
8. one character is sampled from the probability distribution;
9. the sampled character is appended and the four-character window moves forward.

Only the final-position prediction is used during generation.

A local NumPy random generator keeps text generation reproducible.

### Sample the next character

The current four-character context is converted into numerical identifiers and processed by the complete Transformer block.

The Transformer produces logits for all four positions.

During generation, only the logits from the final position are used because that position has access to the complete current context.

Softmax converts these logits into probabilities.

A local random generator samples one identifier from the learned probability distribution and converts it back into a character.

In [17]:
def sample_next_character(
    context,
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    attention_output_weights,
    layer_norm_1_scale,
    layer_norm_1_shift,
    feed_forward_input_weights,
    feed_forward_output_weights,
    layer_norm_2_scale,
    layer_norm_2_shift,
    output_weights,
    random_generator
):
    context_ids = tf.constant([[character_to_id[character] for character in context]], dtype=tf.int32)

    _, _, logits = transformer_forward(
        embedding_matrix,
        position_embedding_matrix,
        query_weights,
        key_weights,
        value_weights,
        attention_output_weights,
        layer_norm_1_scale,
        layer_norm_1_shift,
        feed_forward_input_weights,
        feed_forward_output_weights,
        layer_norm_2_scale,
        layer_norm_2_shift,
        output_weights,
        context_ids
    )

    probabilities = softmax(logits)[0, -1].numpy()

    next_id = random_generator.choice(
        vocabulary_size,
        p=probabilities
    )

    return id_to_character[next_id]

In [18]:
sample_random = np.random.default_rng(SEED)

example_context = "mode"

print("Context:", repr(example_context))

for _ in range(5):
    sampled_character = sample_next_character(
        example_context,
        trained_embedding_matrix,
        trained_position_embedding_matrix,
        trained_query_weights,
        trained_key_weights,
        trained_value_weights,
        trained_attention_output_weights,
        trained_layer_norm_1_scale,
        trained_layer_norm_1_shift,
        trained_feed_forward_input_weights,
        trained_feed_forward_output_weights,
        trained_layer_norm_2_scale,
        trained_layer_norm_2_shift,
        trained_output_weights,
        sample_random
    )

    print("Sampled character:", repr(sampled_character))

Context: 'mode'
Sampled character: 'r'
Sampled character: 'd'
Sampled character: 's'
Sampled character: 'r'
Sampled character: ' '


### Generate text

Text generation starts from a four-character context.

At every step, the Transformer processes the current context and samples the next character from the probability distribution produced by the final position.

The sampled character is appended to the output.

The oldest context character is removed, causing the fixed context window to slide forward by one position.

For example:

`mode -> sampled character`

then:

`ode? -> next sampled character`

and so on.

For every new context window, the complete Transformer computation is performed again.

The model therefore generates text autoregressively while using causal self-attention inside every context window.

In [19]:
def generate_text(
    starting_context,
    number_of_characters,
    embedding_matrix,
    position_embedding_matrix,
    query_weights,
    key_weights,
    value_weights,
    attention_output_weights,
    layer_norm_1_scale,
    layer_norm_1_shift,
    feed_forward_input_weights,
    feed_forward_output_weights,
    layer_norm_2_scale,
    layer_norm_2_shift,
    output_weights,
    seed=42
):
    if len(starting_context) != CONTEXT_LENGTH:
        raise ValueError(
            f"starting_context must contain exactly "
            f"{CONTEXT_LENGTH} characters"
        )

    generated_text = starting_context
    generation_random = np.random.default_rng(seed)

    for _ in range(number_of_characters):
        current_context = generated_text[-CONTEXT_LENGTH:]

        next_character = sample_next_character(
            current_context,
            embedding_matrix,
            position_embedding_matrix,
            query_weights,
            key_weights,
            value_weights,
            attention_output_weights,
            layer_norm_1_scale,
            layer_norm_1_shift,
            feed_forward_input_weights,
            feed_forward_output_weights,
            layer_norm_2_scale,
            layer_norm_2_shift,
            output_weights,
            generation_random
        )

        generated_text += next_character

    return generated_text

In [20]:
generated_text = generate_text(
    starting_context="mode",
    number_of_characters=300,
    embedding_matrix=trained_embedding_matrix,
    position_embedding_matrix=trained_position_embedding_matrix,
    query_weights=trained_query_weights,
    key_weights=trained_key_weights,
    value_weights=trained_value_weights,
    attention_output_weights=trained_attention_output_weights,
    layer_norm_1_scale=trained_layer_norm_1_scale,
    layer_norm_1_shift=trained_layer_norm_1_shift,
    feed_forward_input_weights=trained_feed_forward_input_weights,
    feed_forward_output_weights=trained_feed_forward_output_weights,
    layer_norm_2_scale=trained_layer_norm_2_scale,
    layer_norm_2_shift=trained_layer_norm_2_shift,
    output_weights=trained_output_weights,
    seed=SEED
)

print(generated_text)

moderetr win mawiom n.
sinttl macors mod manes lexper fconged clerning lamamp belers, th comakerincte crsmay rin tleleaserks pamake codvame win mexpl mon tter plett.
e maromp may bextrspectin thin tecoul sy caimp mondinim wing vauang counsiniactl mornd morke pr co thaud ea.
carnd mor ed ne modv textwori


## 5. Tests

These assertions verify the sequential context-target windows, Transformer parameters, causal attention mask, model shapes, probability distributions, gradients, training behavior and reproducibility.

The tests verify that:

- every input and target contains four character identifiers;
- all 13 trainable Transformer variables are TensorFlow variables;
- the model contains exactly 1264 trainable parameters;
- the Transformer produces the expected tensor shapes;
- causal attention prevents access to future positions;
- every attention row sums to 1;
- training and validation sets remain separate;
- training reduces the loss and all losses remain finite;
- next-character probability distributions sum to 1;
- TensorFlow produces finite gradients for every trainable parameter;
- every trainable parameter changes during training;
- retraining with the same seed is reproducible;
- autoregressive generation is reproducible.

In [21]:
assert len(examples) == len(corpus) - CONTEXT_LENGTH
assert len(input_ids) == len(examples)
assert len(target_ids) == len(examples)

assert input_ids.shape == (len(examples), CONTEXT_LENGTH)
assert target_ids.shape == (len(examples), CONTEXT_LENGTH)

assert examples[0][0] == corpus[:CONTEXT_LENGTH]
assert examples[0][1] == corpus[1:CONTEXT_LENGTH + 1]
assert examples[1][0] == corpus[1:1 + CONTEXT_LENGTH]
assert examples[1][1] == corpus[2:2 + CONTEXT_LENGTH]

assert tf.is_tensor(inputs)
assert tf.is_tensor(targets)

# Trainable parameter checks

initial_parameter_variables = [
    embedding_matrix, position_embedding_matrix, query_weights, key_weights, value_weights,
    attention_output_weights, layer_norm_1_scale, layer_norm_1_shift,
    feed_forward_input_weights, feed_forward_output_weights,
    layer_norm_2_scale, layer_norm_2_shift, output_weights
]

trained_parameter_variables = [
    trained_embedding_matrix, trained_position_embedding_matrix,
    trained_query_weights, trained_key_weights, trained_value_weights,
    trained_attention_output_weights, trained_layer_norm_1_scale, trained_layer_norm_1_shift,
    trained_feed_forward_input_weights, trained_feed_forward_output_weights,
    trained_layer_norm_2_scale, trained_layer_norm_2_shift, trained_output_weights
]

assert len(initial_parameter_variables) == 13
assert len(trained_parameter_variables) == 13
assert all(isinstance(variable, tf.Variable) for variable in initial_parameter_variables)
assert all(isinstance(variable, tf.Variable) for variable in trained_parameter_variables)

assert embedding_matrix.shape == (vocabulary_size, EMBEDDING_DIM)
assert position_embedding_matrix.shape == (CONTEXT_LENGTH, EMBEDDING_DIM)
assert query_weights.shape == (EMBEDDING_DIM, ATTENTION_DIM)
assert key_weights.shape == (EMBEDDING_DIM, ATTENTION_DIM)
assert value_weights.shape == (EMBEDDING_DIM, ATTENTION_DIM)
assert attention_output_weights.shape == (ATTENTION_DIM, EMBEDDING_DIM)
assert layer_norm_1_scale.shape == (EMBEDDING_DIM,)
assert layer_norm_1_shift.shape == (EMBEDDING_DIM,)
assert feed_forward_input_weights.shape == (EMBEDDING_DIM, FEED_FORWARD_DIM)
assert feed_forward_output_weights.shape == (FEED_FORWARD_DIM, EMBEDDING_DIM)
assert layer_norm_2_scale.shape == (EMBEDDING_DIM,)
assert layer_norm_2_shift.shape == (EMBEDDING_DIM,)
assert output_weights.shape == (EMBEDDING_DIM, vocabulary_size)

for initial_variable, trained_variable in zip(initial_parameter_variables, trained_parameter_variables):
    assert trained_variable.shape == initial_variable.shape

# Trainable parameter count

expected_trainable_parameters = (
    vocabulary_size * EMBEDDING_DIM
    + CONTEXT_LENGTH * EMBEDDING_DIM
    + 3 * EMBEDDING_DIM * ATTENTION_DIM
    + ATTENTION_DIM * EMBEDDING_DIM
    + 2 * EMBEDDING_DIM
    + EMBEDDING_DIM * FEED_FORWARD_DIM
    + FEED_FORWARD_DIM * EMBEDDING_DIM
    + 2 * EMBEDDING_DIM
    + EMBEDDING_DIM * vocabulary_size
)

assert expected_trainable_parameters == 1264
assert trainable_parameters.numpy() == expected_trainable_parameters

# Forward-pass shape checks

assert example_embeddings.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)
assert example_positioned_embeddings.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)
assert example_queries.shape == (1, CONTEXT_LENGTH, ATTENTION_DIM)
assert example_keys.shape == (1, CONTEXT_LENGTH, ATTENTION_DIM)
assert example_values.shape == (1, CONTEXT_LENGTH, ATTENTION_DIM)
assert example_attention_weights.shape == (1, CONTEXT_LENGTH, CONTEXT_LENGTH)
assert example_transformer_output.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)
assert example_logits.shape == (1, CONTEXT_LENGTH, vocabulary_size)

# Causal attention checks

expected_causal_mask = np.tril(np.ones((CONTEXT_LENGTH, CONTEXT_LENGTH), dtype=np.float32))
assert np.array_equal(causal_mask.numpy(), expected_causal_mask)

attention_test_context = tf.constant(
    [[character_to_id[character] for character in "mode"]],
    dtype=tf.int32
)

attention_test_output, attention_test_weights, attention_test_logits = transformer_forward(
    trained_embedding_matrix, trained_position_embedding_matrix,
    trained_query_weights, trained_key_weights, trained_value_weights,
    trained_attention_output_weights, trained_layer_norm_1_scale, trained_layer_norm_1_shift,
    trained_feed_forward_input_weights, trained_feed_forward_output_weights,
    trained_layer_norm_2_scale, trained_layer_norm_2_shift,
    trained_output_weights, attention_test_context
)

assert attention_test_output.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)
assert attention_test_weights.shape == (1, CONTEXT_LENGTH, CONTEXT_LENGTH)
assert attention_test_logits.shape == (1, CONTEXT_LENGTH, vocabulary_size)

assert np.all(np.isfinite(attention_test_output.numpy()))
assert np.all(np.isfinite(attention_test_weights.numpy()))
assert np.all(np.isfinite(attention_test_logits.numpy()))
assert np.all(attention_test_weights.numpy() >= 0.0)
assert np.all(attention_test_weights.numpy() <= 1.0)

attention_row_sums = tf.reduce_sum(attention_test_weights, axis=-1).numpy()
assert np.allclose(attention_row_sums, np.ones((1, CONTEXT_LENGTH)), atol=1e-6)

future_positions = np.triu(np.ones((CONTEXT_LENGTH, CONTEXT_LENGTH), dtype=bool), k=1)
assert np.allclose(attention_test_weights[0].numpy()[future_positions], 0.0, atol=1e-7)

# Training and validation checks

assert len(train_inputs) + len(validation_inputs) == len(inputs)
assert train_inputs.shape == (len(train_inputs), CONTEXT_LENGTH)
assert train_targets.shape == (len(train_inputs), CONTEXT_LENGTH)
assert validation_inputs.shape == (len(validation_inputs), CONTEXT_LENGTH)
assert validation_targets.shape == (len(validation_inputs), CONTEXT_LENGTH)
assert len(np.intersect1d(train_indices, validation_indices)) == 0

assert len(train_loss_history) == EPOCHS + 1
assert len(validation_loss_history) == EPOCHS + 1
assert final_train_loss < float(initial_train_loss.numpy())
assert final_validation_loss < float(initial_validation_loss.numpy())
assert np.all(np.isfinite(train_loss_history))
assert np.all(np.isfinite(validation_loss_history))
assert best_validation_epoch == int(np.argmin(validation_loss_history))
assert best_validation_loss == validation_loss_history[best_validation_epoch]

# Probability checks

probability_context = tf.constant(
    [[character_to_id[character] for character in "mode"]],
    dtype=tf.int32
)

_, _, probability_logits = transformer_forward(
    trained_embedding_matrix, trained_position_embedding_matrix,
    trained_query_weights, trained_key_weights, trained_value_weights,
    trained_attention_output_weights, trained_layer_norm_1_scale, trained_layer_norm_1_shift,
    trained_feed_forward_input_weights, trained_feed_forward_output_weights,
    trained_layer_norm_2_scale, trained_layer_norm_2_shift,
    trained_output_weights, probability_context
)

probability_values = softmax(probability_logits)[0, -1].numpy()

assert abs(probability_values.sum() - 1.0) < 1e-6
assert np.all(np.isfinite(probability_values))
assert np.all(probability_values >= 0.0)
assert np.all(probability_values <= 1.0)

# Gradient checks

gradient_inputs = train_inputs[:BATCH_SIZE]
gradient_targets = train_targets[:BATCH_SIZE]

with tf.GradientTape() as tape:
    gradient_loss = calculate_loss(
        embedding_matrix, position_embedding_matrix,
        query_weights, key_weights, value_weights,
        attention_output_weights, layer_norm_1_scale, layer_norm_1_shift,
        feed_forward_input_weights, feed_forward_output_weights,
        layer_norm_2_scale, layer_norm_2_shift,
        output_weights, gradient_inputs, gradient_targets
    )

gradients = tape.gradient(gradient_loss, initial_parameter_variables)

assert all(gradient is not None for gradient in gradients)

dense_gradients = [tf.convert_to_tensor(gradient) for gradient in gradients]

for gradient, variable in zip(dense_gradients, initial_parameter_variables):
    assert gradient.shape == variable.shape
    assert np.all(np.isfinite(gradient.numpy()))

# Deterministic retraining

(
    repeated_embedding_matrix, repeated_position_embedding_matrix,
    repeated_query_weights, repeated_key_weights, repeated_value_weights,
    repeated_attention_output_weights, repeated_layer_norm_1_scale, repeated_layer_norm_1_shift,
    repeated_feed_forward_input_weights, repeated_feed_forward_output_weights,
    repeated_layer_norm_2_scale, repeated_layer_norm_2_shift,
    repeated_output_weights, repeated_train_history, repeated_validation_history
) = train_model(
    train_inputs, train_targets, validation_inputs, validation_targets,
    embedding_matrix, position_embedding_matrix,
    query_weights, key_weights, value_weights,
    attention_output_weights, layer_norm_1_scale, layer_norm_1_shift,
    feed_forward_input_weights, feed_forward_output_weights,
    layer_norm_2_scale, layer_norm_2_shift, output_weights,
    learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE,
    epochs=EPOCHS, seed=SEED, print_every=None
)

repeated_parameter_variables = [
    repeated_embedding_matrix, repeated_position_embedding_matrix,
    repeated_query_weights, repeated_key_weights, repeated_value_weights,
    repeated_attention_output_weights, repeated_layer_norm_1_scale, repeated_layer_norm_1_shift,
    repeated_feed_forward_input_weights, repeated_feed_forward_output_weights,
    repeated_layer_norm_2_scale, repeated_layer_norm_2_shift, repeated_output_weights
]

for trained_variable, repeated_variable in zip(trained_parameter_variables, repeated_parameter_variables):
    assert np.allclose(trained_variable.numpy(), repeated_variable.numpy())

assert np.allclose(train_loss_history, repeated_train_history)
assert np.allclose(validation_loss_history, repeated_validation_history)

# Parameter-update checks

for initial_variable, trained_variable in zip(initial_parameter_variables, trained_parameter_variables):
    assert np.max(np.abs(initial_variable.numpy() - trained_variable.numpy())) > 0.0

# Generation reproducibility

first_generation = generate_text(
    starting_context="mode", number_of_characters=30,
    embedding_matrix=trained_embedding_matrix,
    position_embedding_matrix=trained_position_embedding_matrix,
    query_weights=trained_query_weights, key_weights=trained_key_weights, value_weights=trained_value_weights,
    attention_output_weights=trained_attention_output_weights,
    layer_norm_1_scale=trained_layer_norm_1_scale, layer_norm_1_shift=trained_layer_norm_1_shift,
    feed_forward_input_weights=trained_feed_forward_input_weights,
    feed_forward_output_weights=trained_feed_forward_output_weights,
    layer_norm_2_scale=trained_layer_norm_2_scale, layer_norm_2_shift=trained_layer_norm_2_shift,
    output_weights=trained_output_weights, seed=10
)

second_generation = generate_text(
    starting_context="mode", number_of_characters=30,
    embedding_matrix=trained_embedding_matrix,
    position_embedding_matrix=trained_position_embedding_matrix,
    query_weights=trained_query_weights, key_weights=trained_key_weights, value_weights=trained_value_weights,
    attention_output_weights=trained_attention_output_weights,
    layer_norm_1_scale=trained_layer_norm_1_scale, layer_norm_1_shift=trained_layer_norm_1_shift,
    feed_forward_input_weights=trained_feed_forward_input_weights,
    feed_forward_output_weights=trained_feed_forward_output_weights,
    layer_norm_2_scale=trained_layer_norm_2_scale, layer_norm_2_shift=trained_layer_norm_2_shift,
    output_weights=trained_output_weights, seed=10
)

assert first_generation == second_generation
assert len(first_generation) == CONTEXT_LENGTH + 30
assert first_generation.startswith("mode")

print("All checks passed.")

All checks passed.


## Notes

- The model remains a character-level neural language model.
- The training corpus and 27-character vocabulary remain unchanged from Version 8.
- Each example still uses a fixed context of four characters.
- Version 9 changes the targets from one next character to four sequential next-character targets.
- Character identifiers are mapped to trainable 8-dimensional embeddings.
- Trainable positional embeddings preserve the order of the four positions.
- Queries, keys and values use 16-dimensional representations.
- A causal mask prevents each position from attending to future positions.
- The attention output is projected back to the 8-dimensional embedding space.
- A residual connection combines the positioned input with the projected attention output.
- Layer normalization is applied after the first residual connection.
- A feed-forward network expands each position from 8 to 16 values and projects it back to 8 values.
- A second residual connection and layer normalization complete the Transformer block.
- Every context position produces vocabulary logits during training.
- During generation, only the logits from the final position are used to sample the next character.
- The model uses a single attention mechanism rather than multi-head attention.
- No bias terms are used in the attention, feed-forward or vocabulary projections; layer normalization uses trainable scale and shift parameters.
- The model contains 1264 trainable parameters.
- Gradients for all 13 trainable variables are calculated automatically with `tf.GradientTape`.
- Parameters are updated manually with mini-batch gradient descent.
- Training and validation examples remain reproducibly separated.
- Neighboring context windows overlap, so the random example-level train/validation split is not a robust estimate of generalization to unseen text.
- Training loss decreases substantially during training.
- Validation loss improves, reaches a minimum during training and then fluctuates, suggesting mild overfitting.
- The exact training loss, validation loss and best validation epoch are calculated and printed by the notebook for the current execution.
- Small numerical differences may appear between complete executions depending on the TensorFlow execution environment.
- The final epoch parameters are intentionally retained for generation rather than automatically restoring the best validation parameters.
- Generation remains autoregressive and uses a sliding four-character context window.
- The complete Transformer block is recomputed for every new context window.
- The same seed keeps data splitting, mini-batch shuffling, retraining and generation reproducible within the same execution environment.
- The tests verify sequential targets, parameter shapes, causal masking, attention weights, probabilities, gradients, parameter updates and reproducibility.

Version 9 extends the attention mechanism from Version 8 into a complete single Transformer block.

Causal self-attention gives every position access only to itself and its past, making the training procedure consistent with autoregressive generation.

Residual connections, layer normalization and feed-forward processing introduce the main structural components used inside Transformer architectures while keeping the implementation small and explicit.

The model is still intentionally limited to one attention mechanism, one Transformer block, a four-character context and a very small corpus.

Version 9 therefore establishes the basic Transformer architecture and prepares the project for a more complete decoder-only language model in the next version.

Future versions will introduce new components and gradually evolve the architecture.